# Chapter 9 — Concurrency and Parallelism

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 9.

**Notebook environment note:** a Jupyter kernel already has an `asyncio` event loop running in the background (that's how the kernel itself communicates). Because of this, `asyncio.run(main())` — the normal way to start async code from a plain `.py` script — raises `RuntimeError: asyncio.run() cannot be called from a running event loop` if you try it in a notebook cell. Instead, notebooks support **top-level `await`** directly: just write `result = await some_coroutine()` in a cell. The items below use `await` directly for that reason; when you write this same code in a regular script file (not a notebook), wrap your entry point in `asyncio.run(main())` instead, exactly as the book shows.

---


### Item 67 — Use `subprocess` to Manage Child Processes

`subprocess.run(...)` launches another program as a child process, waits for it to finish, and gives you its exit code plus captured stdout/stderr — the standard way to shell out to another command from Python (replacing older, less safe tools like `os.system`).


In [1]:
import subprocess

result = subprocess.run(
    ['echo', 'hello from a child process'],
    capture_output=True, text=True
)
print('exit code:', result.returncode)
print('stdout:', result.stdout.strip())

# A failing command: exit code is nonzero, doesn't raise unless check=True
result2 = subprocess.run(['ls', '/definitely/does/not/exist'], capture_output=True, text=True)
print('exit code:', result2.returncode)
print('stderr:', result2.stderr.strip())


exit code: 0
stdout: hello from a child process
exit code: 2
stderr: ls: cannot access '/definitely/does/not/exist': No such file or directory


### Item 68 — Use Threads for Blocking I/O; Avoid for Parallelism

Python's Global Interpreter Lock (GIL) means only one thread runs Python bytecode at a time — threads do **not** give you true CPU parallelism for compute-heavy work (Item 79 covers what does). What threads ARE good for: blocking I/O (network calls, disk reads) — while one thread waits on I/O, the GIL is released and another thread can run, so multiple slow I/O operations can overlap in wall-clock time even though only one thread executes Python code at any instant.


In [2]:
import time
from threading import Thread

def slow_io_task(n):
    time.sleep(0.2)   # simulates a blocking I/O wait (network call, disk read, etc.)
    return n * n

# Sequential: total time is roughly the SUM of each task's wait
start = time.time()
results = [slow_io_task(i) for i in range(4)]
print('sequential:', results, f'{time.time() - start:.2f}s')

# Threaded: the I/O waits overlap, so total time is roughly the time of ONE task, not four
start = time.time()
results = [None] * 4
threads = []
for i in range(4):
    def worker(index=i):
        results[index] = slow_io_task(index)
    t = Thread(target=worker)
    t.start()
    threads.append(t)
for t in threads:
    t.join()
print('threaded:', results, f'{time.time() - start:.2f}s')


sequential: [0, 1, 4, 9] 0.81s


threaded: [0, 1, 4, 9] 0.21s


### Item 69 — Use `Lock` to Prevent Data Races in Threads

Even though only one thread runs Python bytecode at a time, the GIL can switch between threads *between* individual bytecode instructions — so a supposedly-atomic operation like `counter += 1` (which is actually read-modify-write at the bytecode level) can still be corrupted if two threads interleave on it. `threading.Lock()` (used as `with lock:`) ensures only one thread executes the protected section at a time, eliminating the race.


In [3]:
from threading import Thread, Lock

class Counter:
    def __init__(self):
        self.count = 0
    def increment_unsafe(self):
        self.count += 1   # NOT atomic -- read, add, write are separate steps that can interleave

counter = Counter()
threads = [Thread(target=lambda: [counter.increment_unsafe() for _ in range(10_000)]) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print('unsafe result (often wrong, may vary by run):', counter.count, '-- expected 40000')

class SafeCounter:
    def __init__(self):
        self.count = 0
        self.lock = Lock()
    def increment_safe(self):
        with self.lock:      # only one thread at a time can be inside this block
            self.count += 1

safe_counter = SafeCounter()
threads = [Thread(target=lambda: [safe_counter.increment_safe() for _ in range(10_000)]) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print('safe result (always correct):', safe_counter.count)


unsafe result (often wrong, may vary by run): 40000 -- expected 40000
safe result (always correct): 40000


### Item 70 — Use `Queue` to Coordinate Work Between Threads

`queue.Queue` is a thread-safe FIFO built specifically for handing work items between threads (a "producer" thread adds items with `.put()`, "consumer" threads take them with `.get()`) — its internal locking is handled for you, which is much safer than manually coordinating access to a plain list with your own `Lock`.


In [4]:
from queue import Queue
from threading import Thread

work_queue = Queue()
results = []

def producer():
    for i in range(5):
        work_queue.put(i)   # thread-safe: no manual locking needed
    work_queue.put(None)     # sentinel value telling the consumer "no more work"

def consumer():
    while True:
        item = work_queue.get()
        if item is None:
            break
        results.append(item * item)

producer_thread = Thread(target=producer)
consumer_thread = Thread(target=consumer)
producer_thread.start()
consumer_thread.start()
producer_thread.join()
consumer_thread.join()

print(results)


[0, 1, 4, 9, 16]


### Item 71 — Know How to Recognize When Concurrency Is Necessary

Not every program needs concurrency — it adds real complexity (races, deadlocks, harder debugging). The signal that you genuinely need it: your program has many independent, growing units of work that would otherwise execute one after another and waste time waiting (many slow network calls, many files to process) — a single-threaded loop that's fast enough already doesn't need to become concurrent just because concurrency exists.


In [5]:
import time

def process_item(n):
    time.sleep(0.05)   # stands in for a slow I/O-bound step, e.g. a network request
    return n * 2

items = list(range(20))

# If this many independent, slow, GROWING units of work exist, that's the signal to go concurrent
start = time.time()
results = [process_item(i) for i in items]
sequential_time = time.time() - start
print(f'{len(items)} sequential items took {sequential_time:.2f}s -- this is what concurrency would fix')

# A tiny, one-off task does NOT need this complexity -- concurrency would be overkill here
quick_result = process_item(1)
print('a single quick task needs no concurrency machinery at all:', quick_result)


20 sequential items took 1.05s -- this is what concurrency would fix
a single quick task needs no concurrency machinery at all: 2


### Item 72 — Avoid Creating New `Thread` Instances for On-demand Fan-out

Spawning a brand-new `Thread` for every single unit of work (e.g., one thread per network request, with no upper bound) doesn't scale — thread creation has real overhead, and an unbounded number of threads can exhaust system resources under load. The fix (elaborated across the next few items): use a bounded worker pool instead of ad hoc, uncapped thread creation.


In [6]:
from threading import Thread
import time

def fan_out_unbounded(n_tasks):
    threads = []
    for i in range(n_tasks):
        t = Thread(target=lambda: time.sleep(0.01))
        t.start()             # a NEW thread for every single task -- no cap, no reuse
        threads.append(t)
    for t in threads:
        t.join()
    return len(threads)

count = fan_out_unbounded(50)
print(f'created {count} brand-new threads for just 50 small tasks -- this does not scale')
print('See Item 74 (ThreadPoolExecutor) for the bounded, reusable alternative.')


created 50 brand-new threads for just 50 small tasks -- this does not scale
See Item 74 (ThreadPoolExecutor) for the bounded, reusable alternative.


### Item 73 — Understand How Using `Queue` for Concurrency Requires Refactoring

Retrofitting a `Queue`-based producer/consumer pipeline (Item 70) onto existing sequential code isn't a drop-in change — it usually requires restructuring the code into explicit stages connected by queues, with dedicated worker threads/functions for each stage, plus sentinel values or `Queue.join()`/`task_done()` bookkeeping to know when all the work is truly finished. It's a legitimate pattern, but a bigger refactor than it first appears.


In [7]:
from queue import Queue
from threading import Thread

# A small multi-stage pipeline: this shape (queue between EVERY stage) is the refactor the item warns about
def make_stage(in_queue, out_queue, transform):
    def worker():
        while True:
            item = in_queue.get()
            if item is None:
                out_queue.put(None)   # propagate the "done" signal to the NEXT stage too
                break
            out_queue.put(transform(item))
            in_queue.task_done()
    return Thread(target=worker)

raw_q, doubled_q, final_q = Queue(), Queue(), Queue()

stage1 = make_stage(raw_q, doubled_q, lambda x: x * 2)
stage2 = make_stage(doubled_q, final_q, lambda x: x + 1)
stage1.start(); stage2.start()

for i in range(5):
    raw_q.put(i)
raw_q.put(None)   # signal stage1 there's no more input

stage1.join(); stage2.join()

results = []
while True:
    item = final_q.get()
    if item is None:
        break
    results.append(item)
print(results)   # each stage needed its own queue and coordination -- real restructuring, not a small tweak


[1, 3, 5, 7, 9]


### Item 74 — Consider `ThreadPoolExecutor` When Threads Are Necessary for Concurrency

`concurrent.futures.ThreadPoolExecutor` fixes Item 72's uncapped-thread problem: create it once with a fixed `max_workers`, then `.submit(func, *args)` any number of tasks — the pool reuses a bounded set of worker threads instead of creating a new one per task, and `.submit()` returns a `Future` you can `.result()` on to get the return value (or the raised exception, cleanly propagated).


In [8]:
from concurrent.futures import ThreadPoolExecutor
import time

def slow_square(n):
    time.sleep(0.1)
    return n * n

with ThreadPoolExecutor(max_workers=4) as pool:   # bounded pool -- reuses 4 threads, however many tasks
    futures = [pool.submit(slow_square, i) for i in range(8)]
    results = [f.result() for f in futures]   # .result() blocks until that specific task is done

print(results)

def flaky(n):
    if n == 3:
        raise ValueError('something went wrong for input 3')
    return n

with ThreadPoolExecutor(max_workers=2) as pool:
    future = pool.submit(flaky, 3)
    try:
        future.result()   # the exception raised inside the thread propagates cleanly here
    except ValueError as e:
        print('Propagated exception:', e)


[0, 1, 4, 9, 16, 25, 36, 49]
Propagated exception: something went wrong for input 3


### Item 75 — Achieve Highly Concurrent I/O with Coroutines

`async def` functions (coroutines) let a single thread juggle thousands of concurrent I/O-bound operations far more cheaply than thousands of OS threads would — each `await` point is a place the coroutine voluntarily yields control back to the event loop while it waits, letting other coroutines run in that gap. `asyncio.gather(*coroutines)` runs several coroutines concurrently and waits for all of them.


In [9]:
import asyncio
import time

async def fetch_simulated(n):
    await asyncio.sleep(0.1)   # simulates an async I/O wait -- yields control while waiting
    return n * n

start = time.time()
# gather() runs all of these CONCURRENTLY on a single thread -- not one after another
results = await asyncio.gather(*(fetch_simulated(i) for i in range(10)))
print(results, f'{time.time() - start:.2f}s for 10 concurrent 0.1s waits (not ~1.0s sequential)')


[0, 1, 4, 9, 16, 25, 36, 49, 64, 81] 0.11s for 10 concurrent 0.1s waits (not ~1.0s sequential)


### Item 76 — Know How to Port Threaded I/O to `asyncio`

Moving a threaded I/O design to `asyncio` isn't a mechanical find-and-replace: every function in the call chain that touches an async operation must itself become `async def` and use `await` at each blocking point ("async is contagious") — you can't `await` from inside a plain synchronous function. Planning the conversion means identifying every layer of the call stack that needs to become async, not just the innermost I/O call.


In [10]:
import asyncio

# Original threaded-style (synchronous) call chain
def read_data_sync():
    return 'raw data'

def process_sync():
    data = read_data_sync()
    return data.upper()

print(process_sync())

# Ported to asyncio: EVERY function in the chain that leads to an await must become async
async def read_data_async():
    await asyncio.sleep(0.05)   # the actual async I/O operation
    return 'raw data'

async def process_async():
    data = await read_data_async()   # awaiting requires process_async itself to be async too
    return data.upper()

result = await process_async()
print(result)


RAW DATA
RAW DATA


### Item 77 — Mix Threads and Coroutines to Ease the Transition to `asyncio`

A full-codebase rewrite to `asyncio` all at once is risky — `asyncio.to_thread(blocking_func, *args)` lets you call existing, unmodified blocking/synchronous code from within async code by running it in a background thread and `await`-ing its completion, so you can migrate a codebase incrementally instead of all at once.


In [11]:
import asyncio
import time

def legacy_blocking_function(n):
    time.sleep(0.1)   # existing synchronous code -- not rewritten, still blocking
    return n * 2

async def modern_async_caller():
    # runs the UNCHANGED blocking function in a thread, awaited from async code
    result = await asyncio.to_thread(legacy_blocking_function, 21)
    return result

result = await modern_async_caller()
print(result)   # the legacy function never had to be rewritten as async itself


42


### Item 78 — Maximize Responsiveness of `asyncio` Event Loops with `async`-friendly Worker Threads

If truly blocking work (a CPU-bound calculation, or a library with no async support) must run inside an otherwise-async program, running it directly on the event loop's thread would freeze every other coroutine until it finishes. Offloading it to a worker thread (again via `asyncio.to_thread`, or an explicit `ThreadPoolExecutor` combined with `loop.run_in_executor`) keeps the event loop free to keep servicing other coroutines while that work happens in the background.


In [12]:
import asyncio
import time

def cpu_bound_work():
    total = sum(i * i for i in range(5_000_000))   # a real, blocking computation
    return total

async def responsive_coroutine(tag):
    for i in range(3):
        print(f'{tag}: still responsive, tick {i}')
        await asyncio.sleep(0.05)

async def main():
    # run the blocking work on a separate thread so it does NOT freeze the event loop
    heavy_task = asyncio.create_task(asyncio.to_thread(cpu_bound_work))
    responsive_task = asyncio.create_task(responsive_coroutine('heartbeat'))
    result, _ = await asyncio.gather(heavy_task, responsive_task)
    return result

total = await main()
print('cpu_bound_work result:', total)


heartbeat: still responsive, tick 0


heartbeat: still responsive, tick 1
heartbeat: still responsive, tick 2
cpu_bound_work result: 41666654166667500000


### Item 79 — Consider `concurrent.futures` for True Parallelism

Threads (even pooled) never achieve true CPU parallelism in Python because of the GIL — for genuinely CPU-bound work (heavy computation, not I/O), `concurrent.futures.ProcessPoolExecutor` is the tool: it runs each task in a **separate process**, each with its own Python interpreter and its own GIL, so multiple CPU cores are actually used simultaneously. The tradeoff: data passed to/from worker processes must be picklable, and there's real overhead in starting processes and serializing data across them.


In [13]:
from concurrent.futures import ProcessPoolExecutor
import time

def cpu_heavy(n):
    return sum(i * i for i in range(n))

numbers = [2_000_000] * 4

start = time.time()
sequential_results = [cpu_heavy(n) for n in numbers]
sequential_time = time.time() - start
print(f'sequential: {sequential_time:.2f}s')

start = time.time()
with ProcessPoolExecutor(max_workers=4) as pool:   # separate PROCESSES -- real parallel CPU usage
    parallel_results = list(pool.map(cpu_heavy, numbers))
parallel_time = time.time() - start
print(f'parallel (separate processes): {parallel_time:.2f}s')

print('results match:', sequential_results == parallel_results)
print('Speedup depends on actual CPU core count available -- may be modest on a constrained sandbox.')


sequential: 0.28s


parallel (separate processes): 0.10s
results match: True
Speedup depends on actual CPU core count available -- may be modest on a constrained sandbox.
